# EDA Spending Dashboard 20260422

## Objective
This notebook validates the main analytical outputs behind the ACC102 Personal Spending Insight Dashboard. It focuses on four questions:

1. Is the processed dataset structurally correct?
2. Are the KPI and outlier results reasonable?
3. Do the charts render without error?
4. Does the interpretation layer match the computed metrics?

This notebook is a supporting validation artifact. The main production workflow still runs through `build_artifacts.py` and `app.py`.

## Setup
The notebook is written to run from the `notebooks/` folder or from the project root. It uses the processed dataset generated by `build_artifacts.py`.

In [ ]:
from pathlib import Path
import sys
import asyncio
import warnings

import pandas as pd

if sys.platform.startswith('win') and hasattr(asyncio, 'WindowsSelectorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

warnings.filterwarnings('ignore', category=RuntimeWarning)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.analyzer import build_dashboard_payload
from src.charts import create_category_bar_chart, create_monthly_trend_chart, create_outlier_scatter_chart
from src.narrator import generate_summary_insights, generate_analysis_report

## Load Processed Dataset
The featured intermediate dataset is the closest tabular representation of what the Streamlit app uses for analysis and reporting.

In [ ]:
FEATURED_FILEPATH = PROJECT_ROOT / 'data' / 'processed' / 'transactions_featured_int_20260422.csv'
DF_Featured_int = pd.read_csv(FEATURED_FILEPATH, parse_dates=['DATE'])
DF_Featured_int.head()

## Data Quality Checks
These checks confirm that the processed dataset is large enough, spans enough months, keeps the expected currency, and contains detectable outliers.

In [ ]:
ValidationSummary = {
    'Rows': len(DF_Featured_int),
    'Categories': DF_Featured_int['CATEGORY'].nunique(),
    'Months': DF_Featured_int['YEAR_MONTH'].nunique(),
    'Currencies': ', '.join(DF_Featured_int['CURRENCY'].dropna().astype(str).unique().tolist()),
    'Outliers': int(DF_Featured_int['OUTLIER_FLAG'].sum()),
    'Missing Amount': int(DF_Featured_int['AMOUNT'].isna().sum()),
    'Missing Date': int(DF_Featured_int['DATE'].isna().sum()),
}
DF_ValidationSummary = pd.DataFrame(list(ValidationSummary.items()), columns=['CHECK', 'VALUE'])
DF_ValidationSummary

In [ ]:
assert len(DF_Featured_int) >= 300, 'Expected at least 300 rows.'
assert DF_Featured_int['CATEGORY'].nunique() >= 6, 'Expected at least 6 spending categories.'
assert DF_Featured_int['YEAR_MONTH'].nunique() >= 6, 'Expected at least 6 months of data.'
assert set(DF_Featured_int['CURRENCY'].dropna().unique()) == {'CNY'}, 'Expected the featured dataset currency to be CNY.'
assert int(DF_Featured_int['OUTLIER_FLAG'].sum()) >= 1, 'Expected at least one detectable outlier.'
print('All core validation assertions passed.')

## KPI Payload Validation
The dashboard payload is the structured analytics bundle used by the Streamlit app. This cell checks whether the top-level KPI outputs look reasonable.

In [ ]:
Payload = build_dashboard_payload(DF_Featured_int)
DF_KPI = pd.DataFrame([Payload['kpis']]).T.reset_index()
DF_KPI.columns = ['METRIC', 'VALUE']
DF_KPI

## Spending Structure Snapshot
This section checks which categories dominate spending and how the current sample behaves at the merchant level.

In [ ]:
Payload['category_summary'].assign(CATEGORY_SHARE=lambda df: (df['AMOUNT'] / df['AMOUNT'].sum()).round(4)).head(10)

In [ ]:
Payload['merchant_summary'].head(10)

## Outlier Review
This section validates whether the anomaly detection logic is producing interpretable records rather than empty placeholders.

In [ ]:
DF_OutlierReview = Payload['outliers'][['DATE', 'MERCHANT', 'CATEGORY', 'AMOUNT', 'OUTLIER_REASON']].head(10)
DF_OutlierReview

## Auto-generated Insight Validation
The dashboard includes a short interpretation layer. These outputs should align with the KPI payload and not contradict the tables above.

In [ ]:
generate_summary_insights(DF_Featured_int, Payload)

In [ ]:
DF_AnalysisReport = pd.DataFrame({'Analysis Report': generate_analysis_report(DF_Featured_int, Payload)})
DF_AnalysisReport

## Visual Validation
These charts should render without error and match the tables already reviewed above.

In [ ]:
create_category_bar_chart(Payload['category_summary'])

In [ ]:
create_monthly_trend_chart(Payload['monthly_summary'])

In [ ]:
create_outlier_scatter_chart(DF_Featured_int)

## Conclusion
The notebook confirms that the current featured dataset, payload logic, outlier detection, and visual layer are all consistent with the Streamlit product. If later changes are made to the scoring, outlier, or narration logic, this notebook should be rerun as a lightweight regression check.